In [1]:
!git clone https://github.com/Flex1-tech/Fongbe2French
!unzip -oq "Fongbe2French/fonfr_P0 (2).zip"
%cd fonfr_P0/fonfr2
!pip install -q sentencepiece sacrebleu pyyaml
!ls
!grep time_cap config.yaml

Cloning into 'Fongbe2French'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 87 (delta 0), reused 1 (delta 0), pack-reused 86 (from 1)
Receiving objects: 100% (87/87), 145.24 MiB | 38.79 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (47/47), done.
/kaggle/working/fonfr_P0/fonfr2
config.yaml  data  README.md  requirements.txt	scripts  tokenizers
  time_cap_minutes: 17   # fixé après le calibrage de P0 (= temps du meilleur point x 1,5)


In [2]:
p = "scripts/train.py"
src = open(p, encoding="utf-8").read()
open("scripts/train_P0_original.py", "w", encoding="utf-8").write(src)   # copie de sécurité

# 1) deux nouveaux arguments : --alpha et --sample_target
old = 'ap.add_argument("--device", default="auto")\n'
new = old + '''ap.add_argument("--alpha", type=float, default=0.2, help="P1 : lissage de l'echantillonnage Unigram (petit = plus de variete ; 0 = pas d'echantillonnage, equivaut a P0)")
ap.add_argument("--sample_target", action="store_true", help="P1 : echantillonner aussi le francais (train seulement)")
'''
assert src.count(old) == 1, "ligne --device introuvable"
src = src.replace(old, new)

# 2) l'encodage unique du train est remplace par une fonction appelee a chaque epoch
start = src.index("tr_src = [prepare_source(s) for s in tr.fon]")
end = src.index("def pad(seqs):")
block = '''def encode_train(epoch):
    """P1 : re-segmente tout le train au debut de chaque epoch (echantillonnage Unigram).
    Une graine par epoch => reproductible. Le filtre de longueur est refait a chaque fois,
    sur source ET cible ensemble, pour garder l'alignement."""
    if a.alpha > 0:
        spm.set_random_generator_seed(a.seed * 1000 + epoch)   # encode par LISTE : seul cas reproductible
        src = sp_f.encode(tr.fon.tolist(), enable_sampling=True, nbest_size=-1, alpha=a.alpha)
        tgt = (sp_t.encode(tr.fr.tolist(), enable_sampling=True, nbest_size=-1, alpha=a.alpha)
               if a.sample_target else sp_t.encode(tr.fr.tolist()))
    else:
        src, tgt = sp_f.encode(tr.fon.tolist()), sp_t.encode(tr.fr.tolist())
    keep = [i for i in range(len(tr)) if 0 < len(src[i]) <= MAXS and 0 < len(tgt[i]) <= MAXT]
    return [src[i] for i in keep], [tgt[i] for i in keep]
print(f"train : {len(tr)} paires | val : {len(va)} | appareil : {dev} | alpha : {a.alpha}", flush=True)

'''
src = src[:start] + block + src[end:]

# 3) au debut de chaque epoch : on re-segmente
old = "    te = time.time()\n"
new = old + '''    tr_src, tr_tgt = encode_train(epoch)
    print(f"epoch {epoch} : {len(tr_src)}/{len(tr)} paires gardees", flush=True)
'''
assert src.count(old) == 1, "ligne te = time.time() introuvable"
src = src.replace(old, new)

open(p, "w", encoding="utf-8").write(src)
print("Modification appliquee")

Modification appliquee


In [3]:
%cd /kaggle/working/fonfr_P0/fonfr2
import shutil
p = "scripts/models.py"
src = open(p, encoding="utf-8").read()
if "DeepEncoder" in src:
    print("models.py : A4 deja present, rien a faire")
else:
    shutil.copy(p, "scripts/models_sans_A4.py")
    marker = "class CNNEncoder(nn.Module):"
    assert src.count(marker) == 1, "CNNEncoder introuvable"
    deep = '''class DeepEncoder(nn.Module):
    """A4 : BiLSTM a 2 couches (dropout entre les couches), un 'pont' par couche pour initialiser le decodeur."""
    def __init__(self, vocab, emb, hid, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD)
        self.drop = nn.Dropout(dropout)
        self.rnn = nn.LSTM(emb, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.bridge_h = nn.ModuleList([nn.Linear(2 * hid, hid) for _ in range(2)])
        self.bridge_c = nn.ModuleList([nn.Linear(2 * hid, hid) for _ in range(2)])

    def forward(self, src, src_len):
        x = self.drop(self.emb(src))
        packed = pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, c) = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        # h, c : [4, B, hid] = (couche0 avant, couche0 arriere, couche1 avant, couche1 arriere)
        hs = [torch.tanh(self.bridge_h[l](torch.cat([h[2 * l], h[2 * l + 1]], -1))) for l in range(2)]
        cs = [torch.tanh(self.bridge_c[l](torch.cat([c[2 * l], c[2 * l + 1]], -1))) for l in range(2)]
        return out, (torch.stack(hs, 1), torch.stack(cs, 1))     # etats du decodeur : [B, 2, hid]


'''
    src = src.replace(marker, deep + marker)
    old = '        elif arch == "a6": self.encoder = CNNEncoder('
    assert src.count(old) == 1, "ligne a6 introuvable"
    src = src.replace(old, '        elif arch == "a4": self.encoder = DeepEncoder(src_vocab, emb, hid, drop)\n' + old)
    old = 'raise ValueError(f"architecture inconnue : {arch} (a2, a3, a6)")'
    assert src.count(old) == 1, "message d'erreur introuvable"
    src = src.replace(old, 'raise ValueError(f"architecture inconnue : {arch} (a2, a3, a4, a6)")')
    old = "        self.dec = Cell(emb + enc_dim, hid)\n"
    assert src.count(old) == 1, "ligne self.dec introuvable"
    src = src.replace(old, old + '        self.deep = arch == "a4"\n        if self.deep: self.dec2 = nn.LSTMCell(hid, hid)      # 2e couche du decodeur (A4)\n')
    old = "    def step(self, prev, state, mem):\n        h, c = state\n        e = self.drop(self.emb(prev))\n"
    assert src.count(old) == 1, "debut de step introuvable"
    src = src.replace(old, old + "        if self.deep: return self._step_deep(e, h, c, mem)\n")
    old = "    def forward(self, src, src_len, tgt_in, teacher_forcing=1.0):"
    assert src.count(old) == 1, "forward introuvable"
    step_deep = '''    def _step_deep(self, e, h, c, mem):
        """A4 : decodeur a 2 couches. h, c : [B, 2, hid]. Residuel autour de la 2e couche."""
        ctx, a = self.att(h[:, -1], mem["proj"], mem["enc"], mem["mask"])    # attention sur l'etat de la couche du haut
        h1, c1 = self.dec(torch.cat([e, ctx], -1), (h[:, 0], c[:, 0]))
        h2, c2 = self.dec2(self.drop(h1), (h[:, 1], c[:, 1]))              # dropout entre les couches
        top = h2 + h1                                                      # connexion residuelle
        logits = self.out(self.drop(torch.cat([top, ctx, e], -1)))
        return logits, (torch.stack([h1, h2], 1), torch.stack([c1, c2], 1)), a

'''
    src = src.replace(old, step_deep + old)
    open(p, "w", encoding="utf-8").write(src)
    print("models.py : A4 ajoute")

p = "scripts/train.py"
src = open(p, encoding="utf-8").read()
if 'choices=["a2", "a3", "a4", "a6"]' in src:
    print("train.py : a4 deja autorise")
else:
    old = 'choices=["a2", "a3", "a6"]'
    assert src.count(old) == 1, "choices introuvable dans train.py"
    open(p, "w", encoding="utf-8").write(src.replace(old, 'choices=["a2", "a3", "a4", "a6"]'))
    print("train.py : a4 autorise")

/kaggle/working/fonfr_P0/fonfr2
models.py : A4 ajoute
train.py : a4 autorise


In [4]:
%cd /kaggle/working/fonfr_P0/fonfr2
!python scripts/train.py --pipeline P1 --arch a2 --seed 1 --alpha 0.3 --run_dir /kaggle/working/P1_a2_alpha03_s1
!cd /kaggle/working && zip -rq resultats_P1.zip P1_*
!python scripts/train.py --pipeline P1 --arch a3 --seed 1 --alpha 0.3 --run_dir /kaggle/working/P1_a3_alpha03_s1
!cd /kaggle/working && zip -rq resultats_P1.zip P1_*
!python scripts/train.py --pipeline P1 --arch a4 --seed 1 --alpha 0.3 --run_dir /kaggle/working/P1_a4_alpha03_s1
!cd /kaggle/working && zip -rq resultats_P1.zip P1_*
!python scripts/train.py --pipeline P1 --arch a2 --seed 1 --alpha 0.3 --sample_target --run_dir /kaggle/working/P1_a2_alpha03_cible_s1
!cd /kaggle/working && zip -rq resultats_P1.zip P1_*
!python scripts/train.py --pipeline P1 --arch a2 --seed 1 --alpha 0.2 --run_dir /kaggle/working/P1_a2_alpha02_s1
!cd /kaggle/working && zip -rq resultats_P1.zip P1_*
!python scripts/train.py --pipeline P1 --arch a2 --seed 1 --alpha 0.1 --run_dir /kaggle/working/P1_a2_alpha01_s1
!cd /kaggle/working && zip -rq resultats_P1.zip P1_*

/kaggle/working/fonfr_P0/fonfr2
train : 27730 paires | val : 2242 | appareil : cuda | alpha : 0.3
P1 / a2 / seed 1 : 8,199,072 paramètres
epoch 0 : 27560/27730 paires gardees
epoch 1 : 27553/27730 paires gardees
mise à jour 1000 | epoch 1 | train 5.014 | val 4.650 | chrF  | 1.0 min *meilleur*
epoch 2 : 27555/27730 paires gardees
mise à jour 2000 | epoch 2 | train 4.221 | val 4.351 | chrF  | 1.9 min *meilleur*
epoch 3 : 27556/27730 paires gardees
mise à jour 3000 | epoch 3 | train 3.876 | val 4.204 | chrF 16.36 | 2.8 min *meilleur*
epoch 4 : 27549/27730 paires gardees
mise à jour 4000 | epoch 4 | train 3.666 | val 4.090 | chrF  | 3.7 min *meilleur*
epoch 5 : 27553/27730 paires gardees
mise à jour 5000 | epoch 5 | train 3.522 | val 4.042 | chrF  | 4.6 min *meilleur*
epoch 6 : 27552/27730 paires gardees
mise à jour 6000 | epoch 6 | train 3.382 | val 4.005 | chrF 18.82 | 5.5 min *meilleur*
epoch 7 : 27562/27730 paires gardees
epoch 8 : 27554/27730 paires gardees
mise à jour 7000 | epoch 8 

In [5]:
import glob, subprocess
subprocess.run("cd /kaggle/working && unzip -oq Fongbe2French/Résultats.zip -d resultats_p0", shell=True)
base = glob.glob("/kaggle/working/resultats_p0/*/P0_a2_s1/val_hyp.txt")[0]
r = subprocess.run(["python", "/kaggle/working/fonfr_P0/fonfr2/scripts/evaluate.py",
                    "--hyp", "/kaggle/working/P1_a2_alpha03_s1/val_hyp.txt",
                    "--ref", "/kaggle/working/P1_a2_alpha03_s1/val_ref.txt",
                    "--baseline", base], capture_output=True, text=True)
print(r.stdout, r.stderr)

{
  "bleu": 3.83,
  "chrf": 20.98,
  "n": 2242,
  "baseline": {
    "bleu": 3.77,
    "chrf": 19.71,
    "n": 2242
  },
  "p_gagne_vs_baseline": {
    "bleu": 0.603,
    "chrf": 1.0
  }
}
 
